# lawforge-harvest-v1: Goedel-V2-8B + LoRA adapter harvest pass@K

Load FT v1 adapter (from lawforge-finetune-etp), run pass@K=4 over dev_split (214 problems) + hard2 + hard3. Save accepted Lean tactic bodies to `/kaggle/working/harvested.jsonl` for local mining into cheatsheet patterns.

Adapter location: mounted as kernel source from `pedroafonso2/lawforge-finetune-etp` at `/kaggle/input/lawforge-finetune-etp/adapter/`.

In [ ]:
!pip -q install --force-reinstall --no-deps \
    'torch==2.4.1' 'torchvision==0.19.1' 'torchaudio==2.4.1' \
    --index-url https://download.pytorch.org/whl/cu121
!pip -q install --upgrade 'transformers==4.51.3' accelerate 'bitsandbytes==0.43.3' 'peft==0.12.0'

In [ ]:
import os, subprocess, sys
REPO = '/kaggle/working/lawforge'
if not os.path.isdir(REPO):
    subprocess.check_call(['git', 'clone', '--depth', '1',
                           'https://github.com/PAMF2/lawforge.git', REPO])
sys.path.insert(0, REPO)
print('repo HEAD:',
      subprocess.check_output(['git', '-C', REPO, 'log', '-1', '--oneline']).decode().strip())

print('--- /kaggle/input tree ---')
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.replace('/kaggle/input', '').count(os.sep)
    if depth > 3: continue
    print(root)
    for f_ in files[:10]:
        print(' ', f_)

ADAPTER = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'adapter_config.json' in files:
        ADAPTER = root
        break
assert ADAPTER, 'adapter_config.json not found anywhere in /kaggle/input'
print('ADAPTER resolved to:', ADAPTER)

In [ ]:
import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE = 'Goedel-LM/Goedel-Prover-V2-8B'
K = int(os.environ.get('LAWFORGE_HARVEST_K', '4'))
MAX_TOKENS = int(os.environ.get('LAWFORGE_HARVEST_MAX_TOKENS', '512'))
TEMP = float(os.environ.get('LAWFORGE_HARVEST_TEMP', '0.7'))
TOP_P = float(os.environ.get('LAWFORGE_HARVEST_TOP_P', '0.95'))
LIMIT = int(os.environ.get('LAWFORGE_HARVEST_LIMIT', '0'))

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=bnb_cfg, device_map='cuda', trust_remote_code=True,
)
model = PeftModel.from_pretrained(base, ADAPTER)
model.eval()
print(f'loaded base+adapter K={K} max_tokens={MAX_TOKENS} temp={TEMP} top_p={TOP_P}')
print(f'mem: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
import json
from pathlib import Path

INPUTS = Path(f'{REPO}/kaggle/harvest_v1/inputs')
SPLITS = ['dev_split', 'hard2_test', 'hard3_test']

problems = []
for s in SPLITS:
    path = INPUTS / f'{s}.jsonl'
    if not path.exists():
        print(f'skip {s} (missing)')
        continue
    with path.open() as f:
        for line in f:
            row = json.loads(line)
            row['_split'] = s
            problems.append(row)
if LIMIT > 0:
    problems = problems[:LIMIT]
    print(f'LIMIT cap: {LIMIT}')
print(f'total problems: {len(problems)}')

In [ ]:
LEAN_STMT = (
    'import Mathlib\n'
    'import Aesop\n'
    'set_option maxHeartbeats 400000\n'
    'class Magma (G : Type) where\n'
    '  op : G \u2192 G \u2192 G\n'
    'infixl:70 " \u25c7 " => Magma.op\n\n'
    'theorem sair_implication\n'
    '    (G : Type) [inst : Magma G]\n'
    '    (h : \u2200 x y z w u : G, {eq1})\n'
    '    : \u2200 x y z w u : G, {eq2} := '
)


def to_diamond(s):
    return s.replace('*', '\u25c7') if s else s


def build_prompt(p):
    eq1 = to_diamond(p.get('equation1') or p.get('hypothesis', ''))
    eq2 = to_diamond(p.get('equation2') or p.get('goal', ''))
    return ('Complete the following Lean 4 code:\n\n```lean4\n'
            + LEAN_STMT.format(eq1=eq1, eq2=eq2))


import re
_END = re.compile(r'```')
_TACTIC = re.compile(r'\b(intro|intros|rw|apply|exact|have|simp|aesop|nth_rewrite|symm|cases|refine|repeat|rfl|decide|assumption|calc|conv|try)\b')


def extract_proof(text):
    """Cut at first ``` after `by` and return only the tactic body."""
    s = text.strip()
    if s.startswith('by'):
        body = s
    elif '\nby\n' in s:
        body = 'by' + s.split('\nby\n', 1)[1]
    else:
        body = s
    m = _END.search(body)
    if m:
        body = body[:m.start()]
    return body.strip()


def looks_like_lean(text):
    return bool(_TACTIC.search(text))


print('sample prompt:')
print(build_prompt(problems[0])[-400:])

In [ ]:
import time, json
from pathlib import Path

OUT = Path('/kaggle/working/harvested.jsonl')

@torch.inference_mode()
def sample_k(prompt, k):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=MAX_TOKENS, do_sample=True,
        temperature=TEMP, top_p=TOP_P, num_return_sequences=k,
        pad_token_id=tokenizer.pad_token_id, repetition_penalty=1.05,
    )
    prompt_len = inputs['input_ids'].shape[1]
    return [tokenizer.decode(seq[prompt_len:], skip_special_tokens=True) for seq in out]

t0 = time.time()
kept_total = 0
with OUT.open('w') as out:
    for i, p in enumerate(problems):
        prompt = build_prompt(p)
        try:
            raw = sample_k(prompt, K)
        except Exception as e:
            print(f'[{i+1}/{len(problems)}] FAIL: {type(e).__name__}: {e}')
            raw = []
        proofs = []
        for r in raw:
            body = extract_proof(r)
            if looks_like_lean(body):
                proofs.append(body)
        kept_total += len(proofs)
        out.write(json.dumps({
            'id': p.get('id', ''),
            'split': p.get('_split', ''),
            'eq1': p.get('equation1') or p.get('hypothesis', ''),
            'eq2': p.get('equation2') or p.get('goal', ''),
            'label': p.get('label'),
            'proofs': proofs,
        }) + '\n')
        out.flush()
        if (i+1) % 25 == 0 or i == 0:
            print(f'[{i+1}/{len(problems)}] kept {len(proofs)}/{K}, total {kept_total}, {time.time()-t0:.0f}s')
print(f'done. total proofs kept: {kept_total} from {len(problems)} problems in {time.time()-t0:.0f}s')
print(f'output: {OUT.stat().st_size} bytes')

In [ ]:
import json
with open('/kaggle/working/harvested.jsonl') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        r = json.loads(line)
        print(f"--- {r['id']} ({r['split']}) ---")
        print(f"  eq1: {r['eq1']}")
        print(f"  eq2: {r['eq2']}")
        print(f"  proofs ({len(r['proofs'])}):")
        for p in r['proofs'][:2]:
            print(f"    {p[:200]}")
        print()